In [1]:
import pandas as pd
import numpy as np
from scipy.linalg import block_diag
import os
from datetime import datetime

In [12]:

# --- Build paths relative to current script directory ---
current_dir = os.getcwd()  # gets current working directory (e.g., python_processing)
base_dir = os.path.abspath(os.path.join(current_dir, '..'))  # one level up → Mini_project

iot_dir = os.path.join(base_dir, 'inputs', 'EXIOBASE_IOT_2019_ixi')
inputs_dir = os.path.join(base_dir, 'inputs')
material_dir = os.path.join(iot_dir, 'material')  # nested under EXIOBASE_IOT_2019_ixi
emissions_dir = os.path.join(iot_dir, 'air_emissions')  # nested under EXIOBASE_IOT_2019_ixi
output_path = os.path.join(os.getcwd(), 'intermediate_outputs')


# File paths
C_Sector_Aggregation_path = os.path.join(inputs_dir, 'C_Sector.txt')
Z_path = os.path.join(iot_dir, 'Z.txt')
Y_path = os.path.join(iot_dir, 'Y.txt')
x_path = os.path.join(iot_dir, 'x.txt')  # not used yet, but added for completeness
F_path = os.path.join(material_dir, 'F.txt')
F_Y_path = os.path.join(material_dir, 'F_Y.txt')
F_GHG_path = os.path.join(emissions_dir, 'F.txt')
F_Y_GHG_path = os.path.join(emissions_dir, 'F_Y.txt')

In [3]:
# Generate timestamp
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [4]:
def load_concordance_excel(filepath):
    """
    Load and convert a sector aggregation matrix:
    - First column: detailed sector names
    - First row: aggregated sector names
    - Body: binary values, with blanks/empty treated as 0
    """
    df_raw = pd.read_excel(filepath, header=None)

    # Extract labels
    agg_sector_names = df_raw.iloc[0, 1:].tolist()
    detailed_sector_names = df_raw.iloc[1:, 0].tolist()

    # Extract and clean body
    matrix = df_raw.iloc[1:, 1:]
    matrix = matrix.fillna(0).replace(r'^\s*$', 0, regex=True).astype(int)

    # Sanity check
    assert (matrix.sum(axis=1) == 1).all(), "Each row must contain exactly one '1'."

    # Return matrix with proper labels
    return pd.DataFrame(matrix.values, index=detailed_sector_names, columns=agg_sector_names)

In [5]:
# Calculate the C_Matrix_Sector_Aggregation as a pd.Dataframe for later usage
concordance_path = os.path.join(inputs_dir, 'C_Matrix_Sector_Aggregation.xlsx')
C_Matrix_Sector_Aggregation = load_concordance_excel(concordance_path)

C:\Users\simsam\AppData\Local\Temp\ipykernel_2816\3002079578.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  matrix = matrix.fillna(0).replace(r'^\s*$', 0, regex=True).astype(int)


In [6]:
# Save C_Matrix_Sector_Aggregation as excel to check
# Get current time as string, e.g. '20250526_153045'
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Build filename with timestamp
filename = f'C_Matrix_Sector_Aggregation_cleaned_{timestamp}.xlsx'

# Full output path
output_path = os.path.join(os.getcwd(), 'intermediate_outputs', filename)

# Save
C_Matrix_Sector_Aggregation.to_excel(output_path)

print(f"Saved to: {output_path}")

Saved to: c:\Users\simsam\Desktop\Mini_Project\python_processing\intermediate_outputs\C_Matrix_Sector_Aggregation_cleaned_2025-06-05_18-29-02.xlsx


#### Build C-Block Matrix with 49 times n-sector aggregation

In [7]:
# Wee neeed Y here to extract the regions of EXIOBASE
# --- Load Y.txt ---
Y_path = os.path.join(iot_dir, 'Y.txt')
# Assuming Y is loaded with multiindex columns: (Region, Sector)
Y = pd.read_csv(Y_path, sep='\t', header=[0, 1], index_col=[0, 1])

# Extract region labels from column MultiIndex level 0 (the region level)
region_labels = Y.columns.get_level_values(0).unique().tolist()

print("Extracted regions from Y:", region_labels)

Extracted regions from Y: ['AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK', 'GB', 'US', 'JP', 'CN', 'CA', 'KR', 'BR', 'IN', 'MX', 'RU', 'AU', 'CH', 'TR', 'TW', 'NO', 'ID', 'ZA', 'WA', 'WL', 'WE', 'WF', 'WM']


In [8]:
def build_block_diag_C_with_regions(C_single, region_labels):
    """
    Build block diagonal C matrix with real region names on rows and columns.
    Rows = (region, detailed_sector)
    Columns = (region, aggregated_sector)
    """
    from scipy.linalg import block_diag
    import pandas as pd
    
    num_regions = len(region_labels)
    
    blocks = [C_single.values] * num_regions  # keep as is, no transpose
    C_block = block_diag(*blocks)
    
    row_index = pd.MultiIndex.from_product([region_labels, C_single.index], names=['Region', 'Detailed_Sector'])
    col_index = pd.MultiIndex.from_product([region_labels, C_single.columns], names=['Region', 'Aggregated_Sector'])
    
    return pd.DataFrame(C_block, index=row_index, columns=col_index)

# Usage example:
region_labels = Y.columns.get_level_values(0).unique().tolist()  # Extract from Y columns
C_block_matrix = build_block_diag_C_with_regions(C_Matrix_Sector_Aggregation, region_labels)



In [ ]:
# Define filename and output path
filename = f'C_Block_Matrix_49regions_{timestamp}.xlsx'
output_path_bm = os.path.join(os.getcwd(), 'intermediate_outputs', filename)

# Save to Excel (may take a bit of time because the matrix is large)
C_block_matrix.to_excel(output_path_bm)

print(f"C_block_matrix saved to: {output_path_bm}")

C_block_matrix saved to: c:\Users\simsam\Desktop\Mini_Project\python_processing\intermediate_outputs\C_Block_Matrix_49regions_2025-06-05_18-29-02.xlsx


#### Study the form of the Z matrix of EXIOBASE

In [10]:
# Path already defined
# Z_path = os.path.join(iot_dir, 'Z.txt')

# Load with multi-index on rows and multi-columns on top
#Z = pd.read_csv(Z_path, sep='\t', header=[0, 1], index_col=[0, 1])

# Check shape (should be 7172 x 7172 if 163*44)
print("Z shape:", Z.shape)
print("Row index levels:", Z.index.names)
print("Column index levels:", Z.columns.names)
# Show how many unique regions are in the row index
unique_regions = Z.index.get_level_values(0).unique()
print(f"Number of regions in Z rows: {len(unique_regions)}")
print("Regions:", unique_regions.tolist())
print("-----------------")

print("C shape:", C_block_matrix.shape)
print("Row index levels:", C_block_matrix.index.names)
print("Column index levels:", C_block_matrix.columns.names)
print("C shape:", C_block_matrix.shape)
print("-----------------")

print("F_mat_Y shape:", F_mat_Y_df.shape)
print("F_mat_Y Row index levels:", F_mat_Y_df.index.names)
print("F_mat_Y Column index levels:", F_mat_Y_df.columns.names)
print("-----------------")

print("F_mat shape:", F_mat_df.shape)
print("F_mat Row index levels:", F_mat_df.index.names)
print("F_mat Column index levels:", F_mat_df.columns.names)
print("-----------------")

print("Y_df shape:", Y_df.shape)
print("Y_df Row index levels:", Y_df.index.names)
print("Y_df Column index levels:", Y_df.columns.names)
#F_mat_Y_agg = C_block_matrix.T @ F_mat_Y_df

NameError: name 'Z' is not defined

#### Calculate the Z/Y/F_aggregate matrix 49 times 13 sectors

In [13]:
# --- Load Paths ---
# They are defined in the beginning of the document

# --- Load dataframes ---

Y_df = pd.read_csv(Y_path, sep='\t', header=[0, 1], index_col=[0, 1])
Z_df = pd.read_csv(Z_path, sep='\t', header=[0, 1], index_col=[0, 1])
# material and emission extensions (production AND consumption)
F_mat_df =  pd.read_csv(F_path, sep='\t', header=[0,1], index_col=0)  # Assuming 2-level column header and index_col=0
#F_mat_Y_df =  pd.read_csv(F_Y_path, sep='\t', header=[0,1], index_col=0)  # Assuming 2-level column header and index_col=0
F_GHG_df =  pd.read_csv(F_GHG_path, sep='\t', header=[0,1], index_col=0)  # Assuming 2-level column header and index_col=0
#F_GHG_Y_df =  pd.read_csv(F_Y_GHG_path, sep='\t', header=[0,1], index_col=0)  # Assuming 2-level column header and index_col=0

# --- Matrix operations ---
Y_agg = C_block_matrix.T @ Y_df 
Z_agg = C_block_matrix.T @ Z_df @ C_block_matrix
F_mat_agg = F_mat_df @ C_block_matrix
#F_mat_Y_agg = C_block_matrix.T @ F_mat_Y_df
F_GHG_agg = F_GHG_df @ C_block_matrix
#F_GHG_Y_agg = C_block_matrix.T @ F_GHG_Y_df 
# --- Save with timestamp ---
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

Y_filename = f"Y_aggregated_{timestamp}.xlsx"
Z_filename = f"Z_aggregated_{timestamp}.xlsx"
F_filename1 = f"F_material_aggregated_{timestamp}.xlsx"
#F_filename2 = f"F_material_Y_aggregated_{timestamp}.xlsx"
F_filename3 = f"F_GHG_aggregated_{timestamp}.xlsx"
#F_filename4 = f"F_GHG_Y_aggregated_{timestamp}.xlsx"


Y_agg.to_excel(os.path.join(output_path, Y_filename))
Z_agg.to_excel(os.path.join(output_path, Z_filename))
F_mat_agg.to_excel(os.path.join(output_path, F_filename1))
#F_mat_Y_agg.to_excel(os.path.join(output_path, F_filename2))
F_GHG_agg.to_excel(os.path.join(output_path, F_filename3))
#F_GHG_Y_agg.to_excel(os.path.join(output_path, F_filename4))

print("Files saved:")
print(f"- {Y_filename}")
print(f"- {Z_filename}")
print(f"- {F_filename1}")
#print(f"- {F_filename2}")
print(f"- {F_filename3}")
#print(f"- {F_filename4}")

Files saved:
- Y_aggregated_2025-06-05_18-34-11.xlsx
- Z_aggregated_2025-06-05_18-34-11.xlsx
- F_material_aggregated_2025-06-05_18-34-11.xlsx
- F_GHG_aggregated_2025-06-05_18-34-11.xlsx
